# CNN k=7 robustness evaluation

This notebook performs **inference only** with saved ResNet-50 checkpoints. It evaluates each selected k=7 baseline/augmented pair on the exact same held-out images under three deterministic conditions:

- `clean`: the normal 224 px center-crop evaluation.
- `jpeg60`: JPEG quality 60 with 4:2:0 chroma subsampling.
- `resize_112`: bicubic 224 → 112 → 224.

The degradation is applied after the usual resize/center crop and before ImageNet normalization, to match the ViT robustness test. No weights are updated. Results go to a new output root and never overwrite training artifacts.

In [ ]:
# Run only if packages are missing.
# %pip install -q torch torchvision pandas numpy scikit-learn pillow tqdm

In [ ]:
from __future__ import annotations

import gc
import hashlib
import io
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageFile
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    raise RuntimeError("Select a GPU-backed Jupyter kernel on CARC before evaluation.")
print("GPU:", torch.cuda.get_device_name(0))

## 1. Settings

Defaults evaluate the six ADM k=7 models requested by the teammate: three dataset repeats × baseline/augmented. To evaluate corrected k=7 models for more holdouts later, add their names to `HOLDOUTS`.

In [ ]:
@dataclass
class Config:
    data_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact"
    output_root: str = "/project2/yzhao010_1531/csci566-genimage/results/cnn_robustness_k7_v1"
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 0
    use_amp: bool = True
    threshold: float = 0.5
    calibration_bins: int = 10
    verify_image_hashes: bool = True
    skip_completed: bool = True

CFG = Config()

# The corrected benchmark is searched first; the earlier ADM-pilot layout is also supported.
RUN_ROOTS = [
    "/project2/yzhao010_1531/csci566-genimage/results/cnn_benchmark/notebook/runs",
    "/project2/yzhao010_1531/csci566-genimage/cnn_outputs",
]
HOLDOUTS = ["ADM"]
REPEATS = [0, 1, 2]
VARIANTS = ["baseline", "augmented"]
CONDITIONS = ["clean", "jpeg60", "resize_112"]
SEED = 17

print(asdict(CFG))
print("Run roots:", RUN_ROOTS)
print("Selection:", HOLDOUTS, REPEATS, VARIANTS, CONDITIONS)

## 2. Exact evaluation policy and metrics

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
EVALUATION_POLICY = {
    "version": "cnn_robustness_k7_v1",
    "inference_only": True,
    "geometry": {
        "resize_short_edge": 224,
        "center_crop": 224,
        "interpolation": "bicubic",
    },
    "conditions": {
        "clean": {"operation": None},
        "jpeg60": {"operation": "jpeg_roundtrip", "quality": 60, "subsampling": 2},
        "resize_112": {"operation": "bicubic_resize", "sizes": [224, 112, 224]},
    },
    "operation_order": ["resize_short_edge", "center_crop", "condition", "to_tensor", "imagenet_normalize"],
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "label_mapping": {"real": 0, "fake": 1},
    "decision_threshold": CFG.threshold,
}

def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def file_digest(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def reliability_bins(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    indices = np.minimum((probabilities * bins).astype(int), bins - 1)
    rows = []
    for index in range(bins):
        selected = indices == index
        count = int(selected.sum())
        rows.append({
            "bin": index, "lower": index / bins, "upper": (index + 1) / bins,
            "count": count,
            "mean_prob_fake": float(probabilities[selected].mean()) if count else None,
            "observed_fake_fraction": float(labels[selected].mean()) if count else None,
        })
    return pd.DataFrame(rows)

def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if (labels.ndim != 1 or probabilities.shape != labels.shape or len(labels) == 0
            or not np.isin(labels, [0, 1]).all() or not np.isfinite(probabilities).all()
            or ((probabilities < 0) | (probabilities > 1)).any()):
        raise ValueError("Expected aligned binary labels and finite probabilities in [0, 1].")
    predictions = (probabilities >= CFG.threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    calibration = reliability_bins(labels, probabilities, CFG.calibration_bins)
    occupied = calibration[calibration["count"] > 0]
    ece = (((occupied.mean_prob_fake - occupied.observed_fake_fraction).abs()
            * occupied["count"]).sum() / len(labels))
    clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)
    f1 = float(f1_score(labels, predictions, zero_division=0))
    return {
        "n": int(len(labels)), "accuracy": float(accuracy_score(labels, predictions)),
        "threshold": float(CFG.threshold),
        "precision_fake": float(precision_score(labels, predictions, zero_division=0)),
        "recall_fake": float(recall_score(labels, predictions, zero_division=0)),
        "f1_fake": f1, "f1": f1,
        "auroc": float(roc_auc_score(labels, probabilities)), "ece": float(ece),
        "brier": float(np.mean((probabilities - labels) ** 2)),
        "log_loss": float(-np.mean(labels * np.log(clipped) + (1-labels) * np.log1p(-clipped))),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

## 3. Saved-run discovery and manifest validation

The evaluator accepts both current (`input_test.csv`, run name ending in `__seed17`) and earlier (`test_split.csv`, no seed suffix) CNN run folders. Baseline and augmented models for a given holdout/repeat must contain the same image hashes and labels.

In [ ]:
def locate_run(holdout: str, rep: int, variant: str) -> Path:
    stem = f"holdout_{holdout}_k7_rep{rep}__{variant}"
    checked = []
    for raw_root in RUN_ROOTS:
        root = Path(raw_root)
        candidates = [root / f"{stem}__seed{SEED}", root / stem]
        for candidate in candidates:
            checked.append(str(candidate))
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Saved run not found. Checked:\n - " + "\n - ".join(checked))

def test_csv_for(run_dir: Path) -> Path:
    candidates = [run_dir / "input_test.csv", run_dir / "test_split.csv"]
    found = [path for path in candidates if path.is_file()]
    if len(found) != 1:
        raise FileNotFoundError(f"Expected exactly one saved test manifest in {run_dir}; found {found}")
    return found[0]

def relative_genimage_path(raw_path: str) -> str:
    text = str(raw_path).replace("\\", "/")
    marker = re.search(r"(?:^|/)GenImage_Compact(?:_shared)?/", text)
    if marker:
        return text[marker.end():]
    path = Path(text)
    if not path.is_absolute():
        return path.as_posix()
    try:
        return path.relative_to(Path(CFG.data_root)).as_posix()
    except ValueError as exc:
        raise ValueError(f"Cannot map manifest path to GenImage_Compact: {raw_path}") from exc

def load_test_manifest(run_dir: Path, expected_holdout: str) -> tuple[pd.DataFrame, Path]:
    csv_path = test_csv_for(run_dir)
    frame = pd.read_csv(csv_path)
    required = {"path", "label", "sha256"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns {sorted(missing)}")
    frame = frame.copy()
    frame["path"] = frame.path.astype(str).map(relative_genimage_path)
    frame["label"] = pd.to_numeric(frame.label, errors="raise").astype(int)
    frame["sha256"] = frame.sha256.astype(str).str.lower()
    if len(frame) != 560 or frame.label.value_counts().to_dict() != {0: 280, 1: 280}:
        raise ValueError(f"{csv_path}: expected 560 rows balanced 280 real / 280 fake.")
    if (frame.sha256.duplicated().any() or
            not frame.sha256.str.fullmatch(r"[0-9a-f]{64}").all()):
        raise ValueError(f"{csv_path}: hashes must be unique SHA-256 values.")
    generator_column = "generator" if "generator" in frame else "source_generator"
    if generator_column in frame:
        fake_generators = set(frame.loc[frame.label == 1, generator_column].astype(str))
        if fake_generators != {expected_holdout}:
            raise ValueError(f"{csv_path}: fake test generators {fake_generators} != {expected_holdout}")
    for row in tqdm(frame.itertuples(), total=len(frame), desc=f"verify {run_dir.name}", leave=False):
        image_path = Path(CFG.data_root) / row.path
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        if CFG.verify_image_hashes and file_digest(image_path) != row.sha256:
            raise ValueError(f"Image bytes do not match saved manifest: {image_path}")
    return frame, csv_path

def membership(frame: pd.DataFrame) -> list[tuple[str, int]]:
    return sorted(zip(frame.sha256.astype(str), frame.label.astype(int)))

run_records = []
for holdout in HOLDOUTS:
    for rep in REPEATS:
        paired = {}
        for variant in VARIANTS:
            run_dir = locate_run(holdout, rep, variant)
            checkpoint = run_dir / "best.pt"
            if not checkpoint.is_file():
                raise FileNotFoundError(checkpoint)
            frame, manifest_path = load_test_manifest(run_dir, holdout)
            paired[variant] = frame
            run_records.append({
                "holdout": holdout, "rep": rep, "variant": variant,
                "run_name": run_dir.name, "run_dir": run_dir, "checkpoint": checkpoint,
                "checkpoint_sha256": file_digest(checkpoint),
                "test_manifest": manifest_path, "test_manifest_sha256": file_digest(manifest_path),
                "test_frame": frame,
            })
        reference = membership(paired[VARIANTS[0]])
        for variant in VARIANTS[1:]:
            if membership(paired[variant]) != reference:
                raise ValueError(f"{holdout} rep{rep}: {variant} does not use the paired test images/labels.")

print(f"Preflight passed for {len(run_records)} saved models.")
display(pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in row.items()
                      if k != "test_frame"} for row in run_records]))

## 4. Deterministic transforms, model loading, and prediction

In [ ]:
class RobustnessTransform:
    def __init__(self, condition: str):
        if condition not in CONDITIONS:
            raise ValueError(condition)
        self.condition = condition
        self.geometry = transforms.Compose([
            transforms.Resize(CFG.image_size, interpolation=InterpolationMode.BICUBIC),
            transforms.CenterCrop(CFG.image_size),
        ])
        self.finish = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __call__(self, image: Image.Image) -> torch.Tensor:
        image = self.geometry(image)
        if image.size != (224, 224):
            raise RuntimeError(f"Unexpected post-crop size: {image.size}")
        if self.condition == "jpeg60":
            with io.BytesIO() as buffer:
                image.save(buffer, format="JPEG", quality=60, subsampling=2)
                buffer.seek(0)
                with Image.open(buffer) as decoded:
                    image = decoded.convert("RGB").copy()
        elif self.condition == "resize_112":
            image = image.resize((112, 112), Image.Resampling.BICUBIC)
            image = image.resize((224, 224), Image.Resampling.BICUBIC)
        return self.finish(image)

class ManifestDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, condition: str):
        self.frame = frame.reset_index(drop=True)
        self.transform = RobustnessTransform(condition)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = Path(CFG.data_root) / row.path
        with Image.open(path) as image:
            tensor = self.transform(image.convert("RGB"))
        return {
            "image": tensor, "label": int(row.label),
            "path": str(row.path), "sha256": str(row.sha256),
        }

def make_loader(frame: pd.DataFrame, condition: str):
    return DataLoader(
        ManifestDataset(frame, condition), batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )

def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:  # PyTorch versions before the weights_only argument
        return torch.load(path, map_location="cpu")

def load_model(checkpoint_path: Path) -> nn.Module:
    saved = load_checkpoint(checkpoint_path)
    if not isinstance(saved, dict) or "model_state_dict" not in saved:
        raise ValueError(f"Checkpoint has no model_state_dict: {checkpoint_path}")
    state = saved["model_state_dict"]
    if state and all(key.startswith("module.") for key in state):
        state = {key.removeprefix("module."): value for key, value in state.items()}
    model = resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 1)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    return model

@torch.inference_mode()
def predict(model: nn.Module, frame: pd.DataFrame, condition: str) -> pd.DataFrame:
    rows = []
    for batch in tqdm(make_loader(frame, condition), desc=condition, leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        with autocast(enabled=CFG.use_amp):
            probabilities = torch.sigmoid(model(images).squeeze(1)).float().cpu().numpy()
        for path, image_hash, label, probability in zip(
            batch["path"], batch["sha256"], batch["label"].numpy(), probabilities
        ):
            rows.append({
                "path": path, "sha256": image_hash, "image_id": image_hash,
                "label": int(label), "probability_fake": float(probability),
            })
    result = pd.DataFrame(rows)
    result["prediction"] = (result.probability_fake >= CFG.threshold).astype(int)
    return result

## 5. Run inference

Each condition is resumable. A completed result is reused only when its policy, checkpoint hash, and saved test-manifest hash still match.

In [ ]:
output_root = Path(CFG.output_root)
(output_root / "runs").mkdir(parents=True, exist_ok=True)
policy_path = output_root / "EVALUATION_POLICY.json"
policy_text = json.dumps(EVALUATION_POLICY, indent=2)
if policy_path.exists() and json.loads(policy_path.read_text()) != EVALUATION_POLICY:
    raise ValueError(f"Evaluation policy changed at {policy_path}; use a new output root.")
policy_path.write_text(policy_text)

def clean_source_comparison(run_dir: Path, predictions: pd.DataFrame) -> dict:
    source_path = run_dir / "test_predictions.csv"
    if not source_path.is_file():
        return {"source_predictions_found": False}
    source = pd.read_csv(source_path)
    id_column = "sha256" if "sha256" in source else ("image_id" if "image_id" in source else None)
    if id_column is None or "label" not in source or "probability_fake" not in source:
        return {"source_predictions_found": True, "source_comparison_available": False}
    left = predictions[["sha256", "label", "probability_fake"]].rename(
        columns={"probability_fake": "new_probability"})
    right = source[[id_column, "label", "probability_fake"]].rename(
        columns={id_column: "sha256", "label": "source_label",
                 "probability_fake": "source_probability"})
    joined = left.merge(right, on="sha256", validate="one_to_one")
    if len(joined) != len(predictions) or not (joined.label == joined.source_label).all():
        raise ValueError(f"Existing clean predictions do not match saved test identities/labels: {source_path}")
    return {
        "source_predictions_found": True, "source_comparison_available": True,
        "source_rows": int(len(joined)),
        "max_abs_probability_difference": float(
            np.max(np.abs(joined.new_probability - joined.source_probability))),
    }

def evaluate_record(record: dict, condition: str) -> dict:
    identity = {
        "evaluation_policy": EVALUATION_POLICY, "condition": condition,
        "holdout": record["holdout"], "rep": record["rep"],
        "variant": record["variant"], "run_name": record["run_name"],
        "checkpoint_sha256": record["checkpoint_sha256"],
        "test_manifest_sha256": record["test_manifest_sha256"],
    }
    evaluation_hash = digest_bytes(canonical_json(identity).encode())
    condition_dir = output_root / "runs" / record["run_name"] / condition
    complete_path = condition_dir / "COMPLETE.json"
    result_path = condition_dir / "result.json"
    if CFG.skip_completed and complete_path.is_file() and result_path.is_file():
        marker = json.loads(complete_path.read_text())
        if marker.get("evaluation_hash") != evaluation_hash:
            raise ValueError(f"Stale completion marker at {complete_path}")
        print("Verified completed evaluation; skipping", record["run_name"], condition)
        return json.loads(result_path.read_text())

    condition_dir.mkdir(parents=True, exist_ok=True)
    (condition_dir / "evaluation_config.json").write_text(
        json.dumps({**identity, "evaluation_hash": evaluation_hash}, indent=2))
    model = load_model(record["checkpoint"])
    predictions = predict(model, record["test_frame"], condition)
    predictions["holdout"] = record["holdout"]
    predictions["k"] = 7
    predictions["rep"] = record["rep"]
    predictions["variant"] = record["variant"]
    predictions["condition"] = condition
    predictions["run_name"] = record["run_name"]
    predictions.to_csv(condition_dir / "test_predictions.csv", index=False)
    metrics = binary_metrics(predictions.label, predictions.probability_fake)
    (condition_dir / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
    reliability_bins(predictions.label, predictions.probability_fake, CFG.calibration_bins).to_csv(
        condition_dir / "test_calibration.csv", index=False)
    comparison = clean_source_comparison(record["run_dir"], predictions) if condition == "clean" else {}
    result = {
        "run_name": record["run_name"], "holdout": record["holdout"],
        "k": 7, "rep": int(record["rep"]), "variant": record["variant"],
        "condition": condition, "checkpoint_sha256": record["checkpoint_sha256"],
        "test_manifest_sha256": record["test_manifest_sha256"],
        "evaluation_hash": evaluation_hash, **metrics, **comparison,
    }
    result_path.write_text(json.dumps(result, indent=2))
    complete_path.write_text(json.dumps({"evaluation_hash": evaluation_hash}, indent=2))
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result

all_results = []
for record in run_records:
    print("\n" + "=" * 88)
    print(record["run_name"], "checkpoint", record["checkpoint_sha256"][:12])
    for condition in CONDITIONS:
        all_results.append(evaluate_record(record, condition))

results = pd.DataFrame(all_results).sort_values(["holdout", "rep", "condition", "variant"])
results.to_csv(output_root / "all_results.csv", index=False)
display(results[["holdout", "rep", "variant", "condition", "accuracy", "auroc", "f1_fake"]])

## 6. Comparison tables for the teammate

`paired_variant_deltas.csv` reports augmented minus baseline within the same holdout, repeat, and condition. `robustness_drops_from_clean.csv` reports each degraded score minus that model's clean score.

In [ ]:
metric_columns = ["accuracy", "auroc", "f1_fake", "ece", "brier", "log_loss"]

wide_variant = results.pivot(
    index=["holdout", "rep", "condition"], columns="variant", values=metric_columns
)
if not {"baseline", "augmented"}.issubset(set(results.variant)):
    raise ValueError("Both baseline and augmented results are required for paired deltas.")
deltas = pd.DataFrame(index=wide_variant.index).reset_index()
for metric in metric_columns:
    deltas[f"baseline_{metric}"] = wide_variant[(metric, "baseline")].to_numpy()
    deltas[f"augmented_{metric}"] = wide_variant[(metric, "augmented")].to_numpy()
    deltas[f"delta_{metric}"] = (
        wide_variant[(metric, "augmented")] - wide_variant[(metric, "baseline")]
    ).to_numpy()
deltas.to_csv(output_root / "paired_variant_deltas.csv", index=False)

clean = results[results.condition == "clean"][
    ["holdout", "rep", "variant", *metric_columns]
].rename(columns={metric: f"clean_{metric}" for metric in metric_columns})
degraded = results[results.condition != "clean"].merge(
    clean, on=["holdout", "rep", "variant"], validate="many_to_one")
for metric in metric_columns:
    degraded[f"delta_from_clean_{metric}"] = degraded[metric] - degraded[f"clean_{metric}"]
degraded.to_csv(output_root / "robustness_drops_from_clean.csv", index=False)

aggregate = (
    results.groupby(["condition", "variant"], as_index=False)
    .agg(mean_accuracy=("accuracy", "mean"), std_accuracy=("accuracy", "std"),
         mean_auroc=("auroc", "mean"), std_auroc=("auroc", "std"),
         mean_f1_fake=("f1_fake", "mean"), n_runs=("run_name", "count"))
)
aggregate.to_csv(output_root / "summary_by_condition_variant.csv", index=False)

print("Per-run accuracy and AUROC:")
display(results[["holdout", "rep", "variant", "condition", "accuracy", "auroc"]])
print("Augmented minus baseline (positive accuracy/AUROC is better):")
display(deltas[["holdout", "rep", "condition", "delta_accuracy", "delta_auroc"]])
print("Mean across selected holdouts/repeats:")
display(aggregate)
print("Saved to:", output_root)

## What to report

Send the teammate:

1. The holdout(s) and repeats shown in `all_results.csv`.
2. Accuracy and AUROC for baseline and augmented models under `clean`, `jpeg60`, and `resize_112`.
3. The paired augmented-minus-baseline values from `paired_variant_deltas.csv`.
4. This policy detail: JPEG quality 60, Pillow subsampling value 2 (4:2:0), and bicubic 224 → 112 → 224, each applied to both real and fake images after center cropping and before ImageNet normalization.

Do not describe this as retraining. It is a deterministic inference-only stress test of already trained models.